## 1. Imports and computational environment

This section imports the libraries required for data processing, model training,
optimization, and visualization. The versions of the main dependencies are
reported to facilitate reproducibility.

In [1]:
# =============================================================================
# Standard library
# =============================================================================
import math
import os
import random
import re
import sys

# =============================================================================
# Third-party libraries
# =============================================================================
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.optimize import curve_fit
import sklearn
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

- ### Computational environment:

    - Python      : 3.8.18
    - NumPy       : 1.21.6
    - Matplotlib  : 3.4.2
    - SciPy       : 1.10.1
    - Scikit-learn: 0.24.2
    - TensorFlow  : 2.6.0
    - Keras       : 2.6.0
    - GPU devices : 0

## 2. Experimental Data Loading

The experimental simple-shear data are stored in separate files for each
specimen and fiber orientation. Each file contains three columns:

1. applied shear strain, $\gamma$;
2. shear stress, $\tau_{12}$ [MPa];
3. normal stress, $\sigma_{22}$ [MPa].

The functions below load individual specimens and collect all specimens
associated with a specified fiber orientation.

In [ ]:
def load_shear_data(file_name, base_path):
    """
    Load experimental simple shear data from a single specimen file.

    Each input file is expected to contain three numerical columns:
    shear strain, shear stress, and normal stress. The first row is assumed
    to contain column headers and is therefore skipped.

    Parameters
    ----------
    file_name : str
        Name of the data file to be loaded.

    base_path : str
        Directory containing the experimental data files.

    Returns
    -------
    gamma : numpy.ndarray
        One-dimensional array containing the applied shear strain.

    tau : numpy.ndarray
        One-dimensional array containing the shear stress tau_12 [MPa].

    sigma : numpy.ndarray
        One-dimensional array containing the normal stress sigma_22 [MPa].

    Raises
    ------
    ValueError
        If the loaded file does not contain at least three columns.

    Notes
    -----
    The expected column structure is:

        [gamma, tau_12 (MPa), sigma_22 (MPa)]

    Files are read using ``numpy.loadtxt`` and are assumed to be
    whitespace-delimited. If a different delimiter is used, the
    ``delimiter`` argument should be specified explicitly.
    """
    file_path = os.path.join(base_path, file_name)

    data = np.loadtxt(file_path, skiprows=1)

    # Ensure that the expected three-column structure is present.
    if data.ndim != 2 or data.shape[1] < 3:
        raise ValueError(
            f"File '{file_name}' must contain at least three columns: "
            "[gamma, tau_12, sigma_22]."
        )

    gamma = data[:, 0]
    tau = data[:, 1]
    sigma = data[:, 2]

    return gamma, tau, sigma

In [ ]:
def load_all_shear_samples(
    fiber_direction,
    base_path="./data/pesilox/",
    verbose=True,
):
    """
    Load all experimental specimens associated with a fiber orientation.

    The function searches ``base_path`` for files following the naming
    convention

        data{fiber_direction}_{sample_id}

    for example:

        data0_1
        data0_2
        data45_1
        data90_3

    The files are sorted according to their specimen identifier before
    being loaded.

    Parameters
    ----------
    fiber_direction : int or str
        Fiber orientation identifier used in the file names.
        Typical values are 0, 45, and 90.

    base_path : str, optional
        Directory containing the experimental data files.
        Default is ``"./data/pesilox/"``.

    verbose : bool, optional
        If True, print the files identified for the selected orientation.
        Default is True.

    Returns
    -------
    samples : list of tuple
        List containing one tuple per experimental specimen. Each tuple has
        the form

        ``(gamma, tau, sigma)``

        where each element is a one-dimensional NumPy array.

    Raises
    ------
    FileNotFoundError
        If ``base_path`` does not exist.

    Notes
    -----
    Files that cannot be read are skipped and a warning is displayed.
    """
    if not os.path.isdir(base_path):
        raise FileNotFoundError(
            f"Data directory not found: '{base_path}'"
        )

    fiber_direction = str(fiber_direction)

    # Match files such as data0_1, data45_2, or data90_10.
    pattern = re.compile(
        rf"^data{re.escape(fiber_direction)}_(\d+)$"
    )

    matched_files = []

    for file_name in os.listdir(base_path):
        match = pattern.fullmatch(file_name)

        if match:
            sample_id = int(match.group(1))
            matched_files.append((sample_id, file_name))

    # Sort according to specimen number.
    matched_files.sort(key=lambda item: item[0])

    if verbose:
        print(
            f"Fiber orientation {fiber_direction}°: "
            f"{len(matched_files)} specimen file(s) found."
        )

        for sample_id, file_name in matched_files:
            print(f"  Sample {sample_id}: {file_name}")

    samples = []

    for _, file_name in matched_files:
        try:
            sample = load_shear_data(
                file_name=file_name,
                base_path=base_path,
            )

            samples.append(sample)

        except Exception as exc:
            print(
                f"Warning: could not load '{file_name}'. "
                f"Reason: {exc}"
            )

    return samples

- #### Example of usage:

In [ ]:
# Load all specimens for each fiber orientation.
samples_0 = load_all_shear_samples(
    fiber_direction=0,
    base_path="./data/pesilox/",
)

samples_45 = load_all_shear_samples(
    fiber_direction=45,
    base_path="./data/pesilox/",
)

samples_90 = load_all_shear_samples(
    fiber_direction=90,
    base_path="./data/pesilox/",
)

## 3. Poynting-aware constitutive neural network

The PA-CANN is composed of two additive strain-energy branches:

#### 1. an isotropic, weight-shared Valanis-Landel branch based on the principal
   stretch features

   $
   \xi_{\lambda_i} =
   \left(\lambda_i-\lambda_i^{-1}\right)^2,
   $

   together with the sign-flexible $I_2$ correction;

#### 2. an anisotropic branch that combines orientation-modulated principal-stretch
   features with the pseudo-invariants $I_4$ and $I_5$.

Both branches use the same library of constitutive activation functions.
The anisotropic branch can be activated or deactivated, allowing isotropic
and anisotropic training stages to be handled by the same model.

In [ ]:
class CANNFullyConnected(tf.keras.layers.Layer):
    """
    Rede Neural Totalmente Conectada Acoplada (FCNN).
    Recebe um vetor com todos os invariantes modificados de uma vez
    e cria uma série truncada de produtos e potências cruzadas.
    """
    def __init__(self, num_nodes=18, **kwargs):
        super(CANNFullyConnected, self).__init__(**kwargs)
        # num_nodes deve ser múltiplo de 6 (para dividir igualmente entre as 6 ativações)
        self.num_nodes = num_nodes

    def build(self, input_shape):
        # input_shape[-1] será a quantidade de invariantes (2 se isotrópico, 4 se anisotrópico)
        num_invariants = input_shape[-1]
        initializer = tf.keras.initializers.RandomUniform(minval=0.01, maxval=0.1)
        constraint = tf.keras.constraints.NonNeg()
        
        # W1 mistura TODOS os invariantes em num_nodes ativações
        self.w1 = self.add_weight(
            shape=(num_invariants, self.num_nodes),
            initializer=initializer,
            constraint=constraint,
            trainable=True,
            name="W1_fc"
        )
        
        # W2 soma o resultado de todas as ativações para gerar o escalar de Energia
        self.w2 = self.add_weight(
            shape=(self.num_nodes, 1),
            initializer=initializer,
            constraint=constraint,
            trainable=True,
            name="W2_fc"
        )

    def call(self, inputs):
        # inputs tem shape: (batch_size, num_invariants)
        
        # 1. Mistura Linear (Acoplamento Total)
        # z terá shape (batch_size, num_nodes)
        z = tf.matmul(inputs, self.w1) 
        
        # Dividimos z em 6 grupos para aplicar as diferentes não-linearidades
        u = self.num_nodes // 6
        
        z1 = z[:, 0:u]
        z2 = z[:, u:2*u]
        z3 = z[:, 2*u:3*u]
        z4 = z[:, 3*u:4*u]
        z5 = z[:, 4*u:5*u]
        z6 = z[:, 5*u:6*u]
        
        # 2. Funções de Ativação (As mesmas de sempre, operando nos tensores misturados)
        node1 = z1
        node2 = tf.exp(z2) - 1.0
        node3 = tf.square(z3)
        node4 = tf.exp(tf.square(z4)) - 1.0
        node5 = tf.pow(z5 + 1.0, 0.5) - 1.0
        node6 = tf.pow(tf.square(z6) + 1.0, 0.5) - 1.0
        
        # Junta todas as ativações processadas
        all_nodes = tf.concat([node1, node2, node3, node4, node5, node6], axis=1)
        
        # 3. Combinação Final
        # O resultado é a energia escalar Psi (shape: batch_size, 1)
        psi = tf.matmul(all_nodes, self.w2)
        
        return psi

########################################################################################

class IsotropicValanisLandelCANN(tf.keras.layers.Layer):
    """
    Camada Constitutiva Isotrópica baseada em Estiramentos (Valanis-Landel / Ogden).
    Garante Isotropia Estrita e Estabilidade Termodinâmica via Reparametrização de Poynting.
    """
    def __init__(self, num_nodes=6, epsilon=1e-5, **kwargs):
        super(IsotropicValanisLandelCANN, self).__init__(**kwargs)
        self.num_nodes = num_nodes
        self.epsilon = epsilon

    def build(self, input_shape):
        initializer = tf.keras.initializers.RandomUniform(minval=0.01, maxval=0.1)
        constraint = tf.keras.constraints.NonNeg()
        
        # W1: (1, num_nodes)
        self.w1 = self.add_weight(
            shape=(1, self.num_nodes),
            initializer=initializer,
            constraint=constraint,
            trainable=True,
            name="W1_vl"
        )
        
        # W2: (num_nodes, 1)
        self.w2 = self.add_weight(
            shape=(self.num_nodes, 1),
            initializer=initializer,
            constraint=constraint,
            trainable=True,
            name="W2_vl"
        )

        # Em vez de c2 diretamente, treinamos kappa (sem restrição NonNeg necessária)
        self.kappa = self.add_weight(
            shape=(1, 1),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="kappa_poynting"
        )

    def _get_zeta_vl(self):
        """
        Calcula a derivada c1 = d(psi_vl)/d(xi) na origem (xi = 0).
        """
        u = self.num_nodes // 6
        
        # Derivadas das 6 ativações no ponto z = 0
        d_node1 = tf.ones((1, u))
        d_node2 = tf.ones((1, u))
        d_node3 = tf.zeros((1, u))
        d_node4 = tf.zeros((1, u))
        d_node5 = 0.5 * tf.ones((1, u))
        d_node6 = tf.zeros((1, u))

        # Vetor de derivadas d_activation/dz em z = 0
        d_all = tf.concat([d_node1, d_node2, d_node3, d_node4, d_node5, d_node6], axis=1) # (1, num_nodes)

        # Regra da cadeia: c1 = w1 * d_all * w2
        zeta_vl = tf.matmul(self.w1 * d_all, self.w2) # (1, 1)
        return zeta_vl

    def _eval_single_stretch(self, x_lam):
        z = tf.matmul(x_lam, self.w1)  # (batch_size, num_nodes)
        
        u = self.num_nodes // 6
        z1 = z[:, 0:u]
        z2 = z[:, u:2*u]
        z3 = z[:, 2*u:3*u]
        z4 = z[:, 3*u:4*u]
        z5 = z[:, 4*u:5*u]
        z6 = z[:, 5*u:6*u]
        
        node1 = z1
        node2 = tf.exp(z2) - 1.0
        node3 = tf.square(z3)
        node4 = tf.exp(tf.square(z4)) - 1.0
        node5 = tf.pow(z5 + 1.0, 0.5) - 1.0
        node6 = tf.pow(tf.square(z6) + 1.0, 0.5) - 1.0
        
        all_nodes = tf.concat([node1, node2, node3, node4, node5, node6], axis=1)
        return tf.matmul(all_nodes, self.w2)

    def call(self, lam1, lam2, lam3):
        # 1. Termos de estiramento direto: (lambda - 1/lambda)^2
        x_lam1 = tf.square(lam1 - 1.0 / lam1)
        x_lam2 = tf.square(lam2 - 1.0 / lam2)
        x_lam3 = tf.square(lam3 - 1.0 / lam3)
        
        psi_1 = self._eval_single_stretch(x_lam1)
        psi_2 = self._eval_single_stretch(x_lam2)
        psi_3 = self._eval_single_stretch(x_lam3)

        #S_lambda = x_lam1 + x_lam2 + x_lam3
        #psi_vl = self._eval_single_stretch(S_lambda)
        
        psi_vl = psi_1 + psi_2 + psi_3
        
        # 2. Cálculo dinâmico de c1 e c2
        zeta_vl = self._get_zeta_vl()

        c2 = (
            -2*zeta_vl # -2 aqui
            + tf.nn.softplus(self.kappa)
            + self.epsilon
        )
        
        # 3. Termo de I2 com c2: c2 * (I2 - 3)
        I2_minus_3 = tf.square(lam1**(-1.0)) + tf.square(lam2**(-1.0)) + tf.square(lam3**(-1.0)) - 3.0
        
        # Nota: c2 já pode assumir valores negativos se c1 > softplus(kappa) + eps,
        # portanto somamos + c2 * (I2 - 3)
        psi_i2 = c2 * I2_minus_3
        
        return psi_vl + psi_i2

    @property
    def c2(self):
        """Current value of the explicit Poynting-aware I2 coefficient."""
        zeta_vl = self._get_zeta_vl()
    
        return (
            -2.0 * zeta_vl
            + tf.nn.softplus(self.kappa)
            + self.epsilon
        )
    
    
    @property
    def zeta_vl(self):
        """Initial tangent of the shared VL subnetwork."""
        return self._get_zeta_vl()
    
    
    @property
    def c2_eff_0(self):
        """
        Effective I2 coefficient in the infinitesimal-shear limit:
        c2_eff^0 = zeta_VL + c2.
        """
        zeta_vl = self._get_zeta_vl()
        return zeta_vl + self.c2
    
    
    @property
    def mu0_iso(self):
        """
        Ground-state isotropic shear modulus:
        mu0_iso = 2 * (2*zeta_VL + c2).
        """
        zeta_vl = self._get_zeta_vl()
        return 2.0 * (2.0 * zeta_vl + self.c2)


########################################################################################


class PACANN(tf.keras.Model):
    """
    Modelo Constitutivo Completo Adaptado para Treinamento em Dois Estágios.
    """
    def __init__(self, beta_l1=0.0, **kwargs): 
        super(PACANN, self).__init__(**kwargs)
        # Controla se a contribuição das fibras será calculada no forward pass
        self.aniso_active = False
        self.beta_l1 = float(beta_l1)
        
        # Ambos os ramos são criados obrigatoriamente para preservar os pesos na memória
        self.fc_iso = IsotropicValanisLandelCANN(num_nodes=6, name="fc_network")
        self.fc_aniso = CANNFullyConnected(num_nodes=6, name="fc_aniso")

    def _get_l1_penalty(self):
    
        l1_terms = [
            tf.reduce_sum(tf.abs(self.fc_iso.w1)),
            tf.reduce_sum(tf.abs(self.fc_iso.w2)),
        ]
    
        if self.aniso_active:
            l1_terms.extend([
                tf.reduce_sum(tf.abs(self.fc_aniso.w1)),
                tf.reduce_sum(tf.abs(self.fc_aniso.w2)),
            ])
    
        return self.beta_l1 * tf.add_n(l1_terms)

    def get_psi(self, lam1, lam2, lam3, I4=None, I5=None, alpha=None):
        """Monta a função de energia com acoplamento trigonométrico estrutural"""
        
        # 1. PARTE ISOTRÓPICA
        psi_iso = self.fc_iso(lam1, lam2, lam3)
        
        # 2. PARTE ANISOTRÓPICA (Usando IF nativo do Python)
        if self.aniso_active:
            alpha_rad = alpha * np.pi / 180.0
            x_lam1 = tf.square(lam1 - 1.0/lam1)
            x_lam2 = tf.square(lam2 - 1.0/lam2)
            c = tf.cos(alpha_rad)
            s = tf.sin(alpha_rad)
            
            # Componentes de acoplamento direcional
            sc2 = tf.square(s) * tf.square(c)
            s4c4 = (s**4) + (c**4)
            
            # Recursos baseados no artigo (Matriz x Orientação)
            x_lam1_sc = x_lam1 * sc2
            x_lam1_s4c4 = x_lam1 * s4c4
            x_lam2_sc = x_lam2 * sc2
            x_lam2_s4c4 = x_lam2 * s4c4
            
            # Deformações puras da fibra
            x_I4 = tf.square(I4 - 1.0)
            x_I5 = tf.square(I5 - 1.0)
            
            # Nova entrada da FCNN
            input_aniso = tf.concat([
                x_lam1_sc, x_lam1_s4c4, 
                x_lam2_sc, x_lam2_s4c4, 
                x_I4, x_I5
            ], axis=1)
            
            psi_aniso = self.fc_aniso(input_aniso)
            return psi_iso + psi_aniso
        else:
            return psi_iso


    def call(self, inputs):
        gamma = inputs[:, 0:1]
        alpha = inputs[:, 1:2] 
        gamma_sq = tf.square(gamma)
        root = tf.sqrt(4.0 + gamma_sq)

        # 1. Cinemática Espectral (Matriz)
        lam1 = (gamma + root) / 2.0
        lam2 = (-gamma + root) / 2.0
        lam3 = tf.ones_like(gamma)
    
        # 2. Cinemática Estrutural (Fibras - calculadas apenas se aniso_active=True)
        if self.aniso_active:
            alpha_rad = alpha * np.pi / 180.0
            c = tf.cos(alpha_rad)
            s = tf.sin(alpha_rad)
            
            I4 = c**2 + 2.0*gamma*c*s + (1.0 + gamma_sq)*s**2
            I5 = c**2*(1.0 + gamma_sq) + 2.0*c*s*(2.0*gamma + gamma**3) + s**2*(1.0 + 3.0*gamma_sq + gamma_sq**2)
        else:
            I4, I5 = None, None
    
        # 3. Diferenciação Automática
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([lam1, lam2, lam3])
            if self.aniso_active:
                tape.watch([I4, I5])
            
            psi = self.get_psi(lam1, lam2, lam3, I4, I5, alpha)

        # Derivadas Espectrais da Matriz
        dpsi_dlam1 = tape.gradient(psi, lam1)
        dpsi_dlam2 = tape.gradient(psi, lam2)
        dpsi_dlam3 = tape.gradient(psi, lam3)
        
        # Tensões da Matriz Isotrópica
        sigma22_iso = ((lam1)/(lam1**2+1)) * dpsi_dlam1 + ((lam2)/(lam2**2+1)) * dpsi_dlam2 - dpsi_dlam3
        tau12_iso   = ((lam1**2)/(lam1**2+1)) * dpsi_dlam1 - ((lam2**2)/(lam2**2+1)) * dpsi_dlam2
        
        sigma22_total = sigma22_iso
        tau12_total   = tau12_iso
    
        # 4. Contribuições Anisotrópicas (Se ativadas)
        if self.aniso_active:
            dpsi_dI4 = tape.gradient(psi, I4)
            dpsi_dI5 = tape.gradient(psi, I5)
    
            term_s22_I5 = 4.0 * s * (gamma*c + (1.0 + gamma_sq)*s)
            sigma22_aniso = 2.0 * (s**2) * dpsi_dI4 + term_s22_I5 * dpsi_dI5
    
            term_t12_I4 = 2.0 * (s*c + gamma*(s**2))
            term_t12_I5 = 2.0 * (2.0*(gamma**3 + gamma)*(s**2) + (2.0 + 3.0*gamma_sq)*s*c + gamma)
            tau12_aniso = term_t12_I4 * dpsi_dI4 + term_t12_I5 * dpsi_dI5
    
            sigma22_total += sigma22_aniso
            tau12_total   += tau12_aniso
            
        if self.beta_l1 > 0.0:
            self.add_loss(
                self._get_l1_penalty()
            )
            
        del tape 
        return tf.concat([tau12_total, sigma22_total], axis=1)

In [ ]:
def apply_constitutive_activations(z, num_nodes):
    """
    Apply the constitutive activation-function library.

    The hidden nodes are divided equally into six groups. Each group is
    associated with one activation function:

        f1(z) = z
        f2(z) = exp(z) - 1
        f3(z) = z^2
        f4(z) = exp(z^2) - 1
        f5(z) = sqrt(1 + z) - 1
        f6(z) = sqrt(1 + z^2) - 1

    Parameters
    ----------
    z : tf.Tensor
        Hidden-layer pre-activation tensor with shape
        ``(batch_size, num_nodes)``.

    num_nodes : int
        Total number of hidden nodes. It must be divisible by six.

    Returns
    -------
    tf.Tensor
        Concatenated activation outputs with the same shape as ``z``.

    Raises
    ------
    ValueError
        If ``num_nodes`` is not divisible by six.
    """
    if num_nodes % 6 != 0:
        raise ValueError(
            "'num_nodes' must be divisible by 6 because the architecture "
            "uses six activation-function families."
        )

    z1, z2, z3, z4, z5, z6 = tf.split(
        z,
        num_or_size_splits=6,
        axis=1,
    )

    node1 = z1
    node2 = tf.exp(z2) - 1.0
    node3 = tf.square(z3)
    node4 = tf.exp(tf.square(z4)) - 1.0
    node5 = tf.sqrt(1.0 + z5) - 1.0
    node6 = tf.sqrt(1.0 + tf.square(z6)) - 1.0

    return tf.concat(
        [node1, node2, node3, node4, node5, node6],
        axis=1,
    )


########################################################################################


class CANNFullyConnected(tf.keras.layers.Layer):
    """
    Fully connected constitutive neural-network layer.

    This layer receives all constitutive input features simultaneously.
    A non-negative linear transformation first combines the input features,
    after which the constitutive activation-function library is applied.
    The resulting contributions are finally combined into a scalar
    strain-energy density.

    This layer is used for the anisotropic branch of the PA-CANN, where
    orientation-modulated stretch features and pseudo-invariant features
    may interact within the same nonlinear activation.

    Parameters
    ----------
    num_nodes : int, optional
        Number of hidden constitutive nodes. The value must be divisible
        by six so that all activation-function families receive the same
        number of nodes. Default is 18.

    **kwargs
        Additional keyword arguments passed to ``tf.keras.layers.Layer``.

    Attributes
    ----------
    w1 : tf.Variable
        Non-negative weights mapping constitutive inputs to hidden nodes.

    w2 : tf.Variable
        Non-negative weights mapping activated nodes to the scalar
        strain-energy density.
    """

    def __init__(self, num_nodes=18, **kwargs):
        super().__init__(**kwargs)

        if num_nodes % 6 != 0:
            raise ValueError(
                "'num_nodes' must be divisible by 6."
            )

        self.num_nodes = num_nodes

    def build(self, input_shape):
        """
        Create the trainable non-negative network weights.

        Parameters
        ----------
        input_shape : tf.TensorShape
            Shape of the constitutive input tensor. The final dimension
            corresponds to the number of input features.
        """
        num_features = int(input_shape[-1])

        initializer = tf.keras.initializers.RandomUniform(
            minval=0.01,
            maxval=0.1,
        )

        nonnegative = tf.keras.constraints.NonNeg()

        self.w1 = self.add_weight(
            shape=(num_features, self.num_nodes),
            initializer=initializer,
            constraint=nonnegative,
            trainable=True,
            name="W1_fc",
        )

        self.w2 = self.add_weight(
            shape=(self.num_nodes, 1),
            initializer=initializer,
            constraint=nonnegative,
            trainable=True,
            name="W2_fc",
        )

        super().build(input_shape)

    def call(self, inputs):
        """
        Evaluate the strain-energy contribution of the layer.

        Parameters
        ----------
        inputs : tf.Tensor
            Constitutive feature matrix with shape
            ``(batch_size, num_features)``.

        Returns
        -------
        tf.Tensor
            Scalar strain-energy contribution for each sample, with shape
            ``(batch_size, 1)``.
        """
        z = tf.matmul(inputs, self.w1)

        activated_nodes = apply_constitutive_activations(
            z,
            self.num_nodes,
        )

        return tf.matmul(activated_nodes, self.w2)

        
########################################################################################


class IsotropicValanisLandelCANN(tf.keras.layers.Layer):
    """
    Poynting-aware isotropic Valanis--Landel constitutive layer.

    The isotropic strain-energy density is written as

        psi_iso = psi_VL + c2 * (I2 - 3),

    where ``psi_VL`` is constructed from a shared scalar neural mapping
    applied independently to the three principal-stretch features

        xi_lambda_i = (lambda_i - lambda_i^(-1))^2.

    Weight sharing ensures permutation symmetry with respect to the
    principal stretches.

    The explicit second-invariant coefficient ``c2`` is reparameterized as

        c2 = -2*zeta_VL + softplus(kappa) + epsilon,

    where ``zeta_VL`` is the tangent of the shared Valanis--Landel mapping
    at the reference configuration. This guarantees

        2*zeta_VL + c2 > 0,

    and therefore a positive reference-state isotropic shear modulus while
    retaining sign flexibility in the Poynting response.

    Parameters
    ----------
    num_nodes : int, optional
        Number of constitutive nodes in the shared scalar Valanis--Landel
        mapping. Must be divisible by six. Default is 6.

    epsilon : float, optional
        Small positive constant used in the Poynting-aware
        reparameterization. Default is 1e-5.

    **kwargs
        Additional keyword arguments passed to ``tf.keras.layers.Layer``.
    """

    def __init__(self, num_nodes=6, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)

        if num_nodes % 6 != 0:
            raise ValueError(
                "'num_nodes' must be divisible by 6."
            )

        self.num_nodes = num_nodes
        self.epsilon = float(epsilon)

    def build(self, input_shape):
        """
        Create the shared Valanis--Landel weights and Poynting parameter.
        """
        initializer = tf.keras.initializers.RandomUniform(
            minval=0.01,
            maxval=0.1,
        )

        nonnegative = tf.keras.constraints.NonNeg()

        # Shared input weights for the scalar VL mapping.
        self.w1 = self.add_weight(
            shape=(1, self.num_nodes),
            initializer=initializer,
            constraint=nonnegative,
            trainable=True,
            name="W1_vl",
        )

        # Shared output weights for the scalar VL mapping.
        self.w2 = self.add_weight(
            shape=(self.num_nodes, 1),
            initializer=initializer,
            constraint=nonnegative,
            trainable=True,
            name="W2_vl",
        )

        # Unconstrained parameter used to construct c2.
        self.kappa = self.add_weight(
            shape=(1, 1),
            initializer=tf.keras.initializers.Constant(1.0),
            trainable=True,
            name="kappa_poynting",
        )

        super().build(input_shape)

    def _get_zeta_vl(self):
        """
        Compute the reference-state tangent of the shared VL mapping.

        Returns
        -------
        tf.Tensor
            Scalar quantity

                zeta_VL = dw_VL / d(xi) |_(xi=0),

            with shape ``(1, 1)``.

        Notes
        -----
        The derivatives of the six activation functions at z = 0 are

            [1, 1, 0, 0, 1/2, 0].

        The chain rule is then applied through the shared input and output
        weights.
        """
        units_per_activation = self.num_nodes // 6
        dtype = self.w1.dtype

        ones = tf.ones(
            (1, units_per_activation),
            dtype=dtype,
        )

        zeros = tf.zeros(
            (1, units_per_activation),
            dtype=dtype,
        )

        activation_derivatives = tf.concat(
            [
                ones,
                ones,
                zeros,
                zeros,
                0.5 * ones,
                zeros,
            ],
            axis=1,
        )

        return tf.matmul(
            self.w1 * activation_derivatives,
            self.w2,
        )

    def _eval_single_stretch(self, xi_lambda):
        """
        Evaluate the shared scalar Valanis--Landel mapping.

        Parameters
        ----------
        xi_lambda : tf.Tensor
            Principal-stretch feature

                xi_lambda = (lambda - lambda^(-1))^2,

            with shape ``(batch_size, 1)``.

        Returns
        -------
        tf.Tensor
            Energy contribution associated with the supplied principal
            stretch, with shape ``(batch_size, 1)``.
        """
        z = tf.matmul(xi_lambda, self.w1)

        activated_nodes = apply_constitutive_activations(
            z,
            self.num_nodes,
        )

        return tf.matmul(activated_nodes, self.w2)

    def call(self, lam1, lam2, lam3):
        """
        Evaluate the isotropic PA-CANN strain-energy density.

        Parameters
        ----------
        lam1, lam2, lam3 : tf.Tensor
            Principal stretches, each with shape ``(batch_size, 1)``.

        Returns
        -------
        tf.Tensor
            Isotropic strain-energy density with shape
            ``(batch_size, 1)``.
        """
        # Principal-stretch features.
        xi_lam1 = tf.square(lam1 - 1.0 / lam1)
        xi_lam2 = tf.square(lam2 - 1.0 / lam2)
        xi_lam3 = tf.square(lam3 - 1.0 / lam3)

        # Weight-shared separable Valanis--Landel representation.
        psi_vl = (
            self._eval_single_stretch(xi_lam1)
            + self._eval_single_stretch(xi_lam2)
            + self._eval_single_stretch(xi_lam3)
        )

        # Reference-state tangent of the VL branch.
        zeta_vl = self._get_zeta_vl()

        # Sign-flexible I2 coefficient constrained by positive
        # reference-state isotropic shear stiffness.
        c2 = (
            -2.0 * zeta_vl
            + tf.nn.softplus(self.kappa)
            + self.epsilon
        )

        # For incompressible principal stretches:
        # I2 = lambda_1^(-2) + lambda_2^(-2) + lambda_3^(-2).
        I2_minus_3 = (
            tf.square(tf.math.reciprocal(lam1))
            + tf.square(tf.math.reciprocal(lam2))
            + tf.square(tf.math.reciprocal(lam3))
            - 3.0
        )

        psi_i2 = c2 * I2_minus_3

        return psi_vl + psi_i2

    @property
    def zeta_vl(self):
        """
        Return the reference-state tangent of the shared VL mapping.
        """
        return self._get_zeta_vl()

    @property
    def c2(self):
        """
        Return the current Poynting-aware second-invariant coefficient.
        """
        return (
            -2.0 * self.zeta_vl
            + tf.nn.softplus(self.kappa)
            + self.epsilon
        )

    @property
    def c2_eff_0(self):
        """
        Return the effective I2 coefficient near the reference state.

        The coefficient is defined as

            c2_eff_0 = zeta_VL + c2.
        """
        return self.zeta_vl + self.c2

    @property
    def mu0_iso(self):
        """
        Return the reference-state isotropic shear modulus.

        The modulus is

            mu0_iso = 2 * (2*zeta_VL + c2).
        """
        return 2.0 * (
            2.0 * self.zeta_vl
            + self.c2
        )


########################################################################################


class PACANN(tf.keras.Model):
    """
    Complete Poynting-aware Constitutive Artificial Neural Network.

    The total strain-energy density is additively decomposed as

        psi = psi_iso + psi_aniso,

    where the isotropic branch is represented by a weight-shared
    Valanis--Landel CANN with a Poynting-aware I2 correction, and the
    anisotropic branch uses orientation-modulated stretch features together
    with the pseudo-invariants I4 and I5.

    The anisotropic branch can be switched on or off through
    ``aniso_active``, allowing the same model implementation to be used for
    isotropic and fiber-reinforced datasets.

    Parameters
    ----------
    beta_l1 : float, optional
        L1 regularization coefficient applied to the trainable constitutive
        weights. Default is 0.0.

    **kwargs
        Additional keyword arguments passed to ``tf.keras.Model``.

    Notes
    -----
    The Poynting reparameterization parameter ``kappa`` is not included in
    the L1 penalty.
    """

    def __init__(self, beta_l1=0.0, **kwargs):
        super().__init__(**kwargs)

        self.beta_l1 = float(beta_l1)

        # The anisotropic branch is disabled by default.
        self.aniso_active = False

        self.fc_iso = IsotropicValanisLandelCANN(
            num_nodes=6,
            name="vl_isotropic_branch",
        )

        self.fc_aniso = CANNFullyConnected(
            num_nodes=6,
            name="anisotropic_branch",
        )

    def _get_l1_penalty(self):
        """
        Compute the L1 penalty associated with the active CANN weights.

        Returns
        -------
        tf.Tensor
            Scalar L1 regularization term.

        Notes
        -----
        The isotropic Valanis--Landel weights are always included.
        Anisotropic weights are included only when ``aniso_active=True``.
        The Poynting parameter ``kappa`` is intentionally not regularized.
        """
        l1_terms = [
            tf.reduce_sum(tf.abs(self.fc_iso.w1)),
            tf.reduce_sum(tf.abs(self.fc_iso.w2)),
        ]

        if self.aniso_active:
            l1_terms.extend(
                [
                    tf.reduce_sum(tf.abs(self.fc_aniso.w1)),
                    tf.reduce_sum(tf.abs(self.fc_aniso.w2)),
                ]
            )

        return self.beta_l1 * tf.add_n(l1_terms)

    def get_psi(
        self,
        lam1,
        lam2,
        lam3,
        I4=None,
        I5=None,
        alpha=None,
    ):
        """
        Evaluate the total strain-energy density.

        Parameters
        ----------
        lam1, lam2, lam3 : tf.Tensor
            Principal stretches.

        I4, I5 : tf.Tensor or None, optional
            Anisotropic pseudo-invariants. Required when
            ``aniso_active=True``.

        alpha : tf.Tensor or None, optional
            Fiber orientation angle in degrees. Required when
            ``aniso_active=True``.

        Returns
        -------
        tf.Tensor
            Total strain-energy density with shape ``(batch_size, 1)``.
        """
        psi_iso = self.fc_iso(
            lam1,
            lam2,
            lam3,
        )

        if not self.aniso_active:
            return psi_iso

        if I4 is None or I5 is None or alpha is None:
            raise ValueError(
                "I4, I5, and alpha must be provided when the "
                "anisotropic branch is active."
            )

        degree_to_radian = tf.constant(
            np.pi / 180.0,
            dtype=lam1.dtype,
        )

        alpha_rad = (
            tf.cast(alpha, lam1.dtype)
            * degree_to_radian
        )

        s = tf.sin(alpha_rad)
        c = tf.cos(alpha_rad)

        # Principal-stretch features.
        xi_lam1 = tf.square(
            lam1 - 1.0 / lam1
        )

        xi_lam2 = tf.square(
            lam2 - 1.0 / lam2
        )

        # Orientation functions.
        F1 = tf.square(s) * tf.square(c)
        F2 = tf.pow(s, 4) + tf.pow(c, 4)

        # Orientation-modulated stretch features.
        xi_lam1_F1 = xi_lam1 * F1
        xi_lam1_F2 = xi_lam1 * F2

        xi_lam2_F1 = xi_lam2 * F1
        xi_lam2_F2 = xi_lam2 * F2

        # Pseudo-invariant features.
        xi_I4 = tf.square(I4 - 1.0)
        xi_I5 = tf.square(I5 - 1.0)

        anisotropic_inputs = tf.concat(
            [
                xi_lam1_F1,
                xi_lam1_F2,
                xi_lam2_F1,
                xi_lam2_F2,
                xi_I4,
                xi_I5,
            ],
            axis=1,
        )

        psi_aniso = self.fc_aniso(
            anisotropic_inputs
        )

        return psi_iso + psi_aniso

    def call(self, inputs):
        """
        Predict shear and normal stresses under incompressible simple shear.

        Parameters
        ----------
        inputs : tf.Tensor
            Input tensor with shape ``(batch_size, 2)`` containing

                inputs[:, 0] = shear strain gamma
                inputs[:, 1] = fiber orientation alpha [degrees]

        Returns
        -------
        tf.Tensor
            Predicted stress tensor components with shape
            ``(batch_size, 2)``:

                output[:, 0] = tau_12
                output[:, 1] = sigma_22

        Notes
        -----
        The principal stretches and anisotropic pseudo-invariants are
        specialized to the incompressible simple-shear kinematics used in
        this study.

        Stress components are obtained by automatic differentiation of the
        strain-energy density.
        """
        gamma = inputs[:, 0:1]
        alpha = inputs[:, 1:2]

        gamma_sq = tf.square(gamma)

        # -------------------------------------------------------------
        # Principal stretches under incompressible simple shear
        # -------------------------------------------------------------
        root = tf.sqrt(
            4.0 + gamma_sq
        )

        lam1 = (
            gamma + root
        ) / 2.0

        lam2 = (
            -gamma + root
        ) / 2.0

        lam3 = tf.ones_like(gamma)

        # -------------------------------------------------------------
        # Fiber pseudo-invariants
        # -------------------------------------------------------------
        if self.aniso_active:
            degree_to_radian = tf.constant(
                np.pi / 180.0,
                dtype=gamma.dtype,
            )

            alpha_rad = (
                tf.cast(alpha, gamma.dtype)
                * degree_to_radian
            )

            c = tf.cos(alpha_rad)
            s = tf.sin(alpha_rad)

            I4 = (
                tf.square(c)
                + 2.0 * gamma * c * s
                + (1.0 + gamma_sq) * tf.square(s)
            )

            I5 = (
                tf.square(c) * (1.0 + gamma_sq)
                + 2.0
                * c
                * s
                * (2.0 * gamma + tf.pow(gamma, 3))
                + tf.square(s)
                * (
                    1.0
                    + 3.0 * gamma_sq
                    + tf.square(gamma_sq)
                )
            )

        else:
            I4 = None
            I5 = None

        # -------------------------------------------------------------
        # Automatic differentiation of the strain-energy density
        # -------------------------------------------------------------
        with tf.GradientTape(
            persistent=True
        ) as tape:

            tape.watch(
                [lam1, lam2, lam3]
            )

            if self.aniso_active:
                tape.watch(
                    [I4, I5]
                )

            psi = self.get_psi(
                lam1=lam1,
                lam2=lam2,
                lam3=lam3,
                I4=I4,
                I5=I5,
                alpha=alpha,
            )

        dpsi_dlam1 = tape.gradient(
            psi,
            lam1,
        )

        dpsi_dlam2 = tape.gradient(
            psi,
            lam2,
        )

        dpsi_dlam3 = tape.gradient(
            psi,
            lam3,
        )

        # -------------------------------------------------------------
        # Stress contributions associated with explicit stretch
        # dependence of the total energy
        # -------------------------------------------------------------
        sigma22_stretch = (
            lam1 / (tf.square(lam1) + 1.0)
        ) * dpsi_dlam1 + (
            lam2 / (tf.square(lam2) + 1.0)
        ) * dpsi_dlam2 - dpsi_dlam3

        tau12_stretch = (
            tf.square(lam1)
            / (tf.square(lam1) + 1.0)
        ) * dpsi_dlam1 - (
            tf.square(lam2)
            / (tf.square(lam2) + 1.0)
        ) * dpsi_dlam2

        sigma22_total = sigma22_stretch
        tau12_total = tau12_stretch

        # -------------------------------------------------------------
        # Additional pseudo-invariant stress contributions
        # -------------------------------------------------------------
        if self.aniso_active:
            dpsi_dI4 = tape.gradient(
                psi,
                I4,
            )

            dpsi_dI5 = tape.gradient(
                psi,
                I5,
            )

            sigma22_I5_factor = (
                4.0
                * s
                * (
                    gamma * c
                    + (1.0 + gamma_sq) * s
                )
            )

            sigma22_aniso = (
                2.0
                * tf.square(s)
                * dpsi_dI4
                + sigma22_I5_factor
                * dpsi_dI5
            )

            tau12_I4_factor = (
                2.0
                * (
                    s * c
                    + gamma * tf.square(s)
                )
            )

            tau12_I5_factor = (
                2.0
                * (
                    2.0
                    * (tf.pow(gamma, 3) + gamma)
                    * tf.square(s)
                    + (
                        2.0
                        + 3.0 * gamma_sq
                    )
                    * s
                    * c
                    + gamma
                )
            )

            tau12_aniso = (
                tau12_I4_factor
                * dpsi_dI4
                + tau12_I5_factor
                * dpsi_dI5
            )

            sigma22_total += sigma22_aniso
            tau12_total += tau12_aniso

        # -------------------------------------------------------------
        # L1 regularization
        # -------------------------------------------------------------
        if self.beta_l1 > 0.0:
            self.add_loss(
                self._get_l1_penalty()
            )

        del tape

        return tf.concat(
            [
                tau12_total,
                sigma22_total,
            ],
            axis=1,
        )

## 4. Training and post-processing utilities

This section defines auxiliary routines used for:

- weighted fitting of shear and normal stresses;
- post-training hard pruning;
- tracking the Poynting-aware constitutive parameters during optimization;
- extracting and printing the analytical strain-energy density identified by
  the trained PA-CANN.

The pruning and equation-extraction routines operate only on the constitutive
weights $W_1$ and $W_2$. The Poynting reparameterization variable
$\kappa$ is not pruned.

In [ ]:
def make_weighted_loss(beta_1=1.0, beta_2=10.0):
    """
    Create the weighted mean-squared-error loss used for stress fitting.

    The loss simultaneously penalizes errors in the shear stress tau_12 and
    normal stress sigma_22:

        L_data =
            beta_1 * MSE(tau_12)
            + beta_2 * MSE(sigma_22)

    A larger value of ``beta_2`` can be used to compensate for the typically
    smaller magnitude of the normal-stress response.

    Parameters
    ----------
    beta_1 : float, optional
        Weight assigned to the shear-stress mean-squared error.
        Default is 1.0.

    beta_2 : float, optional
        Weight assigned to the normal-stress mean-squared error.
        Default is 10.0.

    Returns
    -------
    callable
        TensorFlow-compatible loss function accepting ``y_true`` and
        ``y_pred`` with shape ``(batch_size, 2)``.

    Notes
    -----
    The expected stress ordering is

        y[:, 0] = tau_12
        y[:, 1] = sigma_22

    L1 regularization is handled separately through ``model.add_loss`` in the
    PA-CANN model and is therefore not included explicitly in this function.
    """
    beta_1 = float(beta_1)
    beta_2 = float(beta_2)

    def weighted_stress_loss(y_true, y_pred):
        tau12_true = y_true[:, 0:1]
        tau12_pred = y_pred[:, 0:1]

        sigma22_true = y_true[:, 1:2]
        sigma22_pred = y_pred[:, 1:2]

        loss_tau = tf.reduce_mean(
            tf.square(tau12_pred - tau12_true)
        )

        loss_sigma = tf.reduce_mean(
            tf.square(sigma22_pred - sigma22_true)
        )

        return (
            beta_1 * loss_tau
            + beta_2 * loss_sigma
        )

    weighted_stress_loss.__name__ = "weighted_stress_loss"

    return weighted_stress_loss


########################################################################################


def apply_hard_prune(model, epsilon_prune=1e-6):
    """
    Apply post-training hard pruning to the active PA-CANN weights.

    Trainable weights whose absolute magnitude is smaller than
    ``epsilon_prune`` are set exactly to zero. Only weights that are nonzero
    before pruning are counted as newly pruned.

    Parameters
    ----------
    model : PACANN
        Trained PA-CANN model.

    epsilon_prune : float, optional
        Hard-pruning threshold. A weight is removed when

            0 < |W| < epsilon_prune.

        Default is 1e-5.

    Returns
    -------
    report : dict
        Dictionary containing pruning statistics for each constitutive weight
        matrix. For each matrix, the following quantities are reported:

        ``total``
            Total number of weights.

        ``active_before``
            Number of nonzero weights before pruning.

        ``zero_before``
            Number of weights already equal to zero.

        ``pruned``
            Number of active weights removed by the current pruning operation.

        ``active_after``
            Number of nonzero weights remaining after pruning.

    Notes
    -----
    The Poynting reparameterization variable ``kappa`` is not pruned.

    The anisotropic branch is pruned only when ``model.aniso_active=True``.
    """
    epsilon_prune = float(epsilon_prune)

    if epsilon_prune < 0.0:
        raise ValueError(
            "'epsilon_prune' must be non-negative."
        )

    report = {}

    layers_to_prune = {
        "isotropic": model.fc_iso,
    }

    if model.aniso_active:
        layers_to_prune["anisotropic"] = model.fc_aniso

    for layer_name, layer in layers_to_prune.items():

        for weight_name in ("w1", "w2"):

            weights = getattr(
                layer,
                weight_name,
            )

            weights_np = weights.numpy()

            # Weights that were active before pruning.
            active_before_mask = (
                weights_np != 0.0
            )

            # Remove only active weights below the prescribed threshold.
            prune_mask = (
                active_before_mask
                & (
                    np.abs(weights_np)
                    < epsilon_prune
                )
            )

            n_total = weights_np.size
            n_active_before = np.sum(
                active_before_mask
            )

            n_zero_before = (
                n_total
                - n_active_before
            )

            n_pruned = np.sum(
                prune_mask
            )

            pruned_weights = weights_np.copy()

            pruned_weights[
                prune_mask
            ] = 0.0

            weights.assign(
                pruned_weights
            )

            n_active_after = np.sum(
                pruned_weights != 0.0
            )

            report[
                f"{layer_name}_{weight_name}"
            ] = {
                "total": int(n_total),
                "active_before": int(
                    n_active_before
                ),
                "zero_before": int(
                    n_zero_before
                ),
                "pruned": int(
                    n_pruned
                ),
                "active_after": int(
                    n_active_after
                ),
            }

    return report


########################################################################################


class TrackPoyntingParamsCallback(tf.keras.callbacks.Callback):
    """
    Track the Poynting-aware isotropic quantities during training.

    At the end of each epoch, the callback stores:

        zeta_VL
            Reference-state tangent of the shared Valanis--Landel mapping.

        c2
            Explicit second-invariant coefficient.

        c2_eff_0
            Effective I2 coefficient near the reference state,

                c2_eff_0 = zeta_VL + c2.

        mu0_iso
            Reference-state isotropic shear modulus,

                mu0_iso = 2 * (2*zeta_VL + c2).

    Parameters
    ----------
    iso_layer : IsotropicValanisLandelCANN
        Poynting-aware isotropic layer to be monitored.

    Attributes
    ----------
    zeta_vl_history : list of float
        History of zeta_VL.

    c2_history : list of float
        History of c2.

    c2_eff_0_history : list of float
        History of the effective reference-state I2 coefficient.

    mu0_iso_history : list of float
        History of the reference-state isotropic shear modulus.
    """

    def __init__(self, iso_layer):
        super().__init__()

        self.iso_layer = iso_layer

        self.zeta_vl_history = []
        self.c2_history = []
        self.c2_eff_0_history = []
        self.mu0_iso_history = []

    @staticmethod
    def _to_float(value):
        """
        Convert a scalar TensorFlow tensor into a Python float.
        """
        return float(
            tf.squeeze(value).numpy()
        )

    def on_epoch_end(
        self,
        epoch,
        logs=None,
    ):
        """
        Store the current Poynting-aware quantities after each epoch.
        """
        self.zeta_vl_history.append(
            self._to_float(
                self.iso_layer.zeta_vl
            )
        )

        self.c2_history.append(
            self._to_float(
                self.iso_layer.c2
            )
        )

        self.c2_eff_0_history.append(
            self._to_float(
                self.iso_layer.c2_eff_0
            )
        )

        self.mu0_iso_history.append(
            self._to_float(
                self.iso_layer.mu0_iso
            )
        )


########################################################################################


def _format_activation(expression, activation_index):
    """
    Return the symbolic representation of a constitutive activation.

    Parameters
    ----------
    expression : str
        Symbolic representation of the activation argument.

    activation_index : int
        Index of the constitutive activation family:

        0 : z
        1 : exp(z) - 1
        2 : z^2
        3 : exp(z^2) - 1
        4 : sqrt(1 + z) - 1
        5 : sqrt(1 + z^2) - 1

    Returns
    -------
    str
        Formatted symbolic activation.
    """
    templates = (
        lambda z: f"({z})",
        lambda z: f"(exp({z}) - 1)",
        lambda z: f"({z})^2",
        lambda z: f"(exp(({z})^2) - 1)",
        lambda z: f"(sqrt(1 + {z}) - 1)",
        lambda z: f"(sqrt(1 + ({z})^2) - 1)",
    )

    return templates[
        activation_index
    ](expression)


########################################################################################


def _format_linear_combination(
    weights,
    input_names,
):
    """
    Construct a symbolic linear combination from nonzero input weights.

    Parameters
    ----------
    weights : array-like
        Input weights associated with one constitutive node.

    input_names : sequence of str
        Symbolic names of the corresponding constitutive inputs.

    Returns
    -------
    str or None
        Formatted linear combination, or ``None`` if all coefficients
        are exactly zero.

    Notes
    -----
    Only coefficients that are exactly equal to zero are omitted.
    No additional display threshold is applied.
    """
    terms = []

    for coefficient, name in zip(
        weights,
        input_names,
    ):
        if coefficient != 0.0:
            terms.append(
                f"{coefficient:.6g}*{name}"
            )

    if not terms:
        return None

    return " + ".join(terms)


########################################################################################


def print_discovered_equation(model):
    """
    Print the analytical strain-energy density identified by the PA-CANN.

    The function extracts the trained weights from the weight-shared
    isotropic Valanis--Landel branch, the explicit Poynting-aware I2
    correction, and, when active, the anisotropic branch.

    Parameters
    ----------
    model : PACANN
        Trained PA-CANN model.

    Notes
    -----
    Only weights that are exactly equal to zero are omitted from the
    printed expression. No additional threshold is applied.

    If hard pruning is desired, ``apply_hard_prune`` should be called
    before this function.

    The isotropic Valanis-Landel mapping is weight-shared across the
    three principal-stretch features:

        xi_lam_i = (lambda_i - lambda_i^(-1))^2.

    The anisotropic inputs are printed using the notation

        xi_lam1_F1
        xi_lam1_F2
        xi_lam2_F1
        xi_lam2_F2
        xi_I4
        xi_I5

    consistent with the constitutive formulation used in the manuscript.
    """
    equation_terms = []

    # =============================================================
    # 1. Weight-shared isotropic Valanis--Landel branch
    # =============================================================
    iso_layer = model.fc_iso

    w1_iso = iso_layer.w1.numpy()
    w2_iso = iso_layer.w2.numpy()

    num_nodes_iso = w1_iso.shape[1]
    nodes_per_activation = num_nodes_iso // 6

    for node_index in range(num_nodes_iso):

        output_weight = w2_iso[
            node_index,
            0,
        ]

        # Skip only weights that are exactly zero.
        if output_weight == 0.0:
            continue

        input_weight = w1_iso[
            0,
            node_index,
        ]

        if input_weight == 0.0:
            continue

        activation_index = (
            node_index
            // nodes_per_activation
        )

        stretch_terms = []

        for stretch_name in (
            "xi_lam1",
            "xi_lam2",
            "xi_lam3",
        ):
            argument = (
                f"{input_weight:.6g}"
                f"*{stretch_name}"
            )

            stretch_terms.append(
                _format_activation(
                    argument,
                    activation_index,
                )
            )

        vl_expression = " + ".join(
            stretch_terms
        )

        equation_terms.append(
            f"{output_weight:.6g}"
            f" * [ {vl_expression} ]"
        )

    # =============================================================
    # 2. Explicit Poynting-aware I2 correction
    # =============================================================
    c2_value = float(
        tf.squeeze(
            iso_layer.c2
        ).numpy()
    )

    if c2_value != 0.0:

        if c2_value < 0.0:
            equation_terms.append(
                f"- {abs(c2_value):.6g}"
                " * (I2 - 3)"
            )

        else:
            equation_terms.append(
                f"{c2_value:.6g}"
                " * (I2 - 3)"
            )

    # =============================================================
    # 3. Anisotropic branch
    # =============================================================
    if model.aniso_active:

        aniso_layer = model.fc_aniso

        w1_aniso = (
            aniso_layer.w1.numpy()
        )

        w2_aniso = (
            aniso_layer.w2.numpy()
        )

        anisotropic_inputs = (
            "xi_lam1_F1",
            "xi_lam1_F2",
            "xi_lam2_F1",
            "xi_lam2_F2",
            "xi_I4",
            "xi_I5",
        )

        num_nodes_aniso = (
            w1_aniso.shape[1]
        )

        nodes_per_activation = (
            num_nodes_aniso // 6
        )

        for node_index in range(
            num_nodes_aniso
        ):

            output_weight = w2_aniso[
                node_index,
                0,
            ]

            # Skip only weights that are exactly zero.
            if output_weight == 0.0:
                continue

            activation_index = (
                node_index
                // nodes_per_activation
            )

            argument = (
                _format_linear_combination(
                    weights=w1_aniso[
                        :,
                        node_index,
                    ],
                    input_names=anisotropic_inputs,
                )
            )

            # If all incoming weights are exactly zero,
            # this node contributes nothing.
            if argument is None:
                continue

            activated_expression = (
                _format_activation(
                    argument,
                    activation_index,
                )
            )

            equation_terms.append(
                f"{output_weight:.6g}"
                f" * {activated_expression}"
            )

    # =============================================================
    # 4. Print the complete identified energy
    # =============================================================
    print("=" * 68)
    print(
        "Discovered PA-CANN strain-energy density"
    )
    print("=" * 68)

    if equation_terms:

        print(
            "Psi = "
            + " +\n      ".join(
                equation_terms
            )
        )

    else:
        print(
            "Psi = 0.0"
        )

    print("=" * 68)


########################################################################################


def prepare_training_data(
    angles,
    gamma_max=0.6,
    base_path="./data/pesilox/",
):
    """
    Prepare experimental data for PA-CANN training.

    For each material configuration or fiber orientation, all experimental
    specimen curves are interpolated onto a common shear-strain grid.
    The pointwise mean shear and normal stresses are used as training targets.

    Experimental standard deviations are also computed on the same grid and
    returned separately for subsequent model evaluation, but they are not
    included in the training targets.

    Parameters
    ----------
    angles : sequence
        Material configurations included in the training dataset.

        Use ``"iso"`` for the isotropic material and numerical values
        such as ``0``, ``45``, and ``90`` for fiber orientations in degrees.

        Examples
        --------
        ``["iso"]``

        ``[0, 45, 90]``

    gamma_max : float, optional
        Maximum shear strain retained from the experimental curves.
        Default is 0.6.

    base_path : str, optional
        Directory containing the experimental specimen files.
        Default is ``"./data/pesilox/"``.

    Returns
    -------
    X_train : numpy.ndarray
        Model inputs with shape ``(n_points, 2)``. The columns contain

            0 : shear strain gamma
            1 : fiber orientation alpha [degrees]

        For isotropic data, alpha is set to zero as a placeholder because
        the anisotropic branch is disabled.

    y_train : numpy.ndarray
        Mean experimental stress targets with shape ``(n_points, 2)``:

            0 : mean shear stress tau_12 [MPa]
            1 : mean normal stress sigma_22 [MPa]

    experimental_data : dict
        Dictionary containing the common shear-strain grid, mean stresses,
        and specimen-to-specimen standard deviations for each configuration.
        These quantities can be used later for plotting and computation of
        scatter-normalized errors.
    """
    all_gamma = []
    all_alpha = []
    all_targets = []

    experimental_data = {}

    for alpha in angles:

        dataset = load_all_shear_samples(
            fiber_direction=alpha,
            base_path=base_path,
        )

        if len(dataset) == 0:
            raise ValueError(
                f"No experimental samples were found for configuration "
                f"'{alpha}'."
            )

        # Number of available points up to gamma_max in the shortest
        # experimental curve.
        min_length = min(
            np.sum(
                np.asarray(sample[0]).reshape(-1)
                <= gamma_max
            )
            for sample in dataset
        )

        min_length = int(min_length)

        if min_length < 2:
            raise ValueError(
                f"Insufficient data points below gamma={gamma_max} "
                f"for configuration '{alpha}'."
            )

        # Common interpolation grid for all specimens of this configuration.
        gamma_common = np.linspace(
            0.0,
            gamma_max,
            min_length,
            dtype=np.float32,
        )

        tau_curves = []
        sigma_curves = []

        # -------------------------------------------------------------
        # Interpolate each experimental specimen onto the common grid
        # -------------------------------------------------------------
        for gamma_raw, tau_raw, sigma_raw in dataset:

            gamma_raw = np.asarray(
                gamma_raw
            ).reshape(-1)

            tau_raw = np.asarray(
                tau_raw
            ).reshape(-1)

            sigma_raw = np.asarray(
                sigma_raw
            ).reshape(-1)

            mask = gamma_raw <= gamma_max

            gamma_filtered = gamma_raw[mask]
            tau_filtered = tau_raw[mask]
            sigma_filtered = sigma_raw[mask]

            tau_interp = np.interp(
                gamma_common,
                gamma_filtered,
                tau_filtered,
            )

            sigma_interp = np.interp(
                gamma_common,
                gamma_filtered,
                sigma_filtered,
            )

            tau_curves.append(
                tau_interp
            )

            sigma_curves.append(
                sigma_interp
            )

        tau_curves = np.asarray(
            tau_curves,
            dtype=np.float32,
        )

        sigma_curves = np.asarray(
            sigma_curves,
            dtype=np.float32,
        )

        # -------------------------------------------------------------
        # Pointwise experimental means used for training
        # -------------------------------------------------------------
        tau_mean = np.mean(
            tau_curves,
            axis=0,
        )

        sigma_mean = np.mean(
            sigma_curves,
            axis=0,
        )

        # -------------------------------------------------------------
        # Experimental scatter used only for subsequent evaluation
        # -------------------------------------------------------------
        if len(dataset) > 1:

            tau_std = np.std(
                tau_curves,
                axis=0,
                ddof=1,
            )

            sigma_std = np.std(
                sigma_curves,
                axis=0,
                ddof=1,
            )

        else:
            tau_std = np.full_like(
                tau_mean,
                np.nan,
            )

            sigma_std = np.full_like(
                sigma_mean,
                np.nan,
            )

        gamma_exp = gamma_common.reshape(
            -1,
            1,
        )

        # alpha is unused when the anisotropic branch is inactive.
        alpha_value = (
            0.0
            if alpha == "iso"
            else float(alpha)
        )

        alpha_exp = np.full_like(
            gamma_exp,
            alpha_value,
            dtype=np.float32,
        )

        # Training targets contain only the experimental mean stresses.
        target = np.column_stack(
            [
                tau_mean,
                sigma_mean,
            ]
        ).astype(np.float32)

        all_gamma.append(
            gamma_exp
        )

        all_alpha.append(
            alpha_exp
        )

        all_targets.append(
            target
        )

        experimental_data[alpha] = {
            "gamma": gamma_common,
            "tau_mean": tau_mean,
            "sigma_mean": sigma_mean,
            "tau_std": tau_std,
            "sigma_std": sigma_std,
            "tau_curves": tau_curves,
            "sigma_curves": sigma_curves,
        }

    gamma_train = np.vstack(
        all_gamma
    )

    alpha_train = np.vstack(
        all_alpha
    )

    X_train = np.hstack(
        [
            gamma_train,
            alpha_train,
        ]
    ).astype(np.float32)

    y_train = np.vstack(
        all_targets
    ).astype(np.float32)

    return (
        X_train,
        y_train,
        experimental_data,
    )

## 5. Training the PA-CANN

The example below trains the PA-CANN for the selected material configurations.
For isotropic-only training, the anisotropic branch is disabled. When one or
more fiber-reinforced configurations are included, the anisotropic branch is
activated automatically.

In [ ]:
# =============================================================================
# Training configuration
# =============================================================================

ANGLES = [45]

GAMMA_MAX = 0.6

BETA_1 = 1.0
BETA_2 = 10.0
BETA_L1 = 1e-5
LEARNING_RATE = 1e-3

EPOCHS = 100000

X_train, y_train, experimental_data = prepare_training_data(
    angles=ANGLES,
    gamma_max=GAMMA_MAX,
    base_path="./data/pesilox/",
)

print(f"Training inputs : {X_train.shape}")
print(f"Training targets: {y_train.shape}")

In [ ]:
# =============================================================================
# Initialize the PA-CANN
# =============================================================================

tf.keras.backend.clear_session()

model = PACANN(
    beta_l1=BETA_L1,
)

# Activate the anisotropic branch whenever fiber-reinforced data are present.
is_isotropic_only = (
    len(ANGLES) == 1
    and ANGLES[0] == "iso"
)

model.aniso_active = not is_isotropic_only

print(
    "Anisotropic branch:",
    "active" if model.aniso_active else "inactive",
)

# =============================================================================
# Compile the model
# =============================================================================

loss_fn = make_weighted_loss(
    beta_1=BETA_1,
    beta_2=BETA_2,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
)

poynting_tracker = TrackPoyntingParamsCallback(
    model.fc_iso
)

# =============================================================================
# Train the PA-CANN
# =============================================================================

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    callbacks=[
        poynting_tracker,
    ],
    verbose=0,
)

print("Training completed.")
print(
    f"Final loss: "
    f"{history.history['loss'][-1]:.6e}"
)

- ### Post-training pruning and constitutive equation extraction

Hard pruning is applied only after optimization. The pruning threshold does not
participate in the training procedure. After pruning, the remaining nonzero
weights define the analytical constitutive equation reported by the PA-CANN.

In [ ]:
pruning_report = apply_hard_prune(
    model,
    epsilon_prune=1e-6,
)

pruning_report

## 6. Post-processing and result export

This section evaluates the trained PA-CANN, visualizes the optimization
history and predicted stress responses, and stores the numerical results
required for subsequent analysis and figure reproduction.

Experimental standard deviations are retrieved from the data-preparation step
and therefore correspond to the same common shear-strain grid used to compute
the experimental mean curves.

In [ ]:
print_discovered_equation(model)

In [ ]:
# =============================================================================
# Configuration selected for post-processing
# =============================================================================

CONFIGURATION = 45
# Examples:
# CONFIGURATION = "iso"
# CONFIGURATION = 0
# CONFIGURATION = 45
# CONFIGURATION = 90

results = experimental_data[CONFIGURATION]

gamma_plot = results["gamma"]
tau_mean = results["tau_mean"]
sigma_mean = results["sigma_mean"]
tau_std = results["tau_std"]
sigma_std = results["sigma_std"]

alpha_value = (
    0.0
    if CONFIGURATION == "iso"
    else float(CONFIGURATION)
)

X_eval = np.column_stack(
    [
        gamma_plot,
        np.full_like(
            gamma_plot,
            alpha_value,
            dtype=np.float32,
        ),
    ]
).astype(np.float32)

predictions = model(
    X_eval,
    training=False,
).numpy()

tau_pred = predictions[:, 0]
sigma_pred = predictions[:, 1]

- ### Plots:

In [ ]:
# =============================================================================
# Training-loss evolution
# =============================================================================

loss_history = np.asarray(
    history.history["loss"],
    dtype=np.float64,
)

fig, ax = plt.subplots(
    figsize=(9, 3),
)

ax.semilogy(
    loss_history,
    color="#d62728",
    linewidth=2.0,
    label="PA-CANN",
)

ax.set_xlabel(
    "Epoch",
    fontsize=14,
)

ax.set_ylabel(
    "Loss",
    fontsize=14,
)

ax.set_title(
    "Training Loss Evolution",
    fontsize=15,
    fontweight="bold",
)

ax.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.5,
)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=12,
)

plt.tight_layout()

plt.savefig(
    "training_loss_example.pdf",
    format="pdf",
    bbox_inches="tight",
)

plt.show()

- #### Poynting-aware branch evolution:

The following plot monitors the evolution of the main quantities associated
with the Poynting-aware isotropic branch during training:

- $\zeta_{\mathrm{VL}}$: coefficient associated with the shared
  Valanis-Landel stretch contribution;
- $c_2$: explicit $I_2$-based correction controlling the sign flexibility
  of the Poynting response;
- $\mu_{0,\mathrm{iso}}/2 = 2\zeta_{\mathrm{VL}} + c_2$: half of the
  reference-state isotropic shear modulus.

Tracking these quantities during optimization is useful to verify that the
reference-state stability condition remains satisfied throughout training.

In [ ]:
def plot_poynting_branch_evolution(
    tracker,
    save_path=None,
    y_limits=(-2.0, 2.0),
):
    """
    Plot the training evolution of the Poynting-aware branch parameters.

    Parameters
    ----------
    tracker : TrackPoyntingParamsCallback
        Callback instance used during training to store the histories of
        zeta_VL, c2, c2_eff_0, and mu0_iso.

    save_path : str or None, optional
        Output path for saving the figure. If None, the figure is not saved.

    y_limits : tuple of float, optional
        Vertical-axis limits for the plot. Default is (-2.0, 2.0).

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure object.

    ax : matplotlib.axes.Axes
        Axes object.

    Notes
    -----
    The plotted quantity

        mu0_iso / 2 = 2 * zeta_VL + c2

    must remain positive to satisfy the imposed positive reference-state
    isotropic shear-stiffness condition.
    """
    zeta_history = np.asarray(
        tracker.zeta_vl_history,
        dtype=float,
    )

    c2_history = np.asarray(
        tracker.c2_history,
        dtype=float,
    )

    mu0_half_history = (
        0.5
        * np.asarray(
            tracker.mu0_iso_history,
            dtype=float,
        )
    )

    epochs = np.arange(
        1,
        len(zeta_history) + 1,
    )

    fig, ax = plt.subplots(
        figsize=(9, 4)
    )

    ax.plot(
        epochs,
        zeta_history,
        label=r'$\zeta_{\mathrm{VL}}$',
        color='#1f77b4',
        linewidth=1.6,
    )

    ax.plot(
        epochs,
        c2_history,
        label=r'$c_2$',
        color='#d62728',
        linestyle='--',
        linewidth=1.6,
    )

    ax.plot(
        epochs,
        mu0_half_history,
        label=(
            r'$\mu_{0,\mathrm{iso}}/2'
            r'=2\zeta_{\mathrm{VL}}+c_2>0$'
        ),
        color='#2ca02c',
        linestyle='-.',
        linewidth=1.6,
    )

    ax.axhline(
        0.0,
        color='black',
        linewidth=0.8,
        linestyle=':',
    )

    ax.set_xlabel(
        'Epoch',
        fontsize=18,
    )

    ax.set_ylabel(
        'Parameter value',
        fontsize=18,
    )

    ax.set_title(
        'Poynting-Aware Branch Evolution',
        fontsize=16,
        fontweight='bold',
    )

    ax.tick_params(
        axis='both',
        labelsize=16,
    )

    ax.grid(
        True,
        alpha=0.22,
    )

    ax.legend(
        loc='upper right',
        frameon=True,
        fontsize=14,
    )

    if y_limits is not None:
        ax.set_ylim(y_limits)

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            format='pdf',
            dpi=300,
            bbox_inches='tight',
        )

    plt.show()

    return fig, ax

plot_poynting_branch_evolution(
    tracker=poynting_tracker,
    save_path=(
        f"poynting_aware_{CONFIGURATION}_example.pdf"
    ),
    y_limits=(-2.0, 2.0),
)

In [ ]:
# =============================================================================
# Stress-response comparison
# =============================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

# -------------------------------------------------------------------------
# Shear stress
# -------------------------------------------------------------------------
axes[0].scatter(
    gamma_plot,
    tau_mean,
    color="#333333",
    s=18,
    alpha=0.8,
    label="Experimental mean",
)

axes[0].fill_between(
    gamma_plot,
    tau_mean - tau_std,
    tau_mean + tau_std,
    color="#333333",
    alpha=0.2,
    label=r"Experimental $\pm 1$ SD",
)

axes[0].plot(
    gamma_plot,
    tau_pred,
    color="#d62728",
    linewidth=2.5,
    label="PA-CANN",
)

axes[0].set_xlabel(
    r"$\gamma$",
    fontsize=16,
)

axes[0].set_ylabel(
    r"$\tau_{12}$ [MPa]",
    fontsize=16,
)

axes[0].set_title(
    "Shear Stress",
    fontsize=17,
    fontweight="bold",
)

axes[0].set_xlim(
    0.0,
    GAMMA_MAX,
)

axes[0].set_ylim(
    -0.03,
    0.9,
)

axes[0].grid(
    True,
    linestyle="--",
    alpha=0.5,
)

axes[0].legend(
    fontsize=12,
)

axes[0].tick_params(
    axis="both",
    labelsize=12,
)

# -------------------------------------------------------------------------
# Normal stress
# -------------------------------------------------------------------------
axes[1].scatter(
    gamma_plot,
    sigma_mean,
    color="#333333",
    s=18,
    alpha=0.8,
    label="Experimental mean",
)

axes[1].fill_between(
    gamma_plot,
    sigma_mean - sigma_std,
    sigma_mean + sigma_std,
    color="#333333",
    alpha=0.2,
    label=r"Experimental $\pm 1$ SD",
)

axes[1].plot(
    gamma_plot,
    sigma_pred,
    color="#d62728",
    linewidth=2.5,
    label="PA-CANN",
)

axes[1].set_xlabel(
    r"$\gamma$",
    fontsize=16,
)

axes[1].set_ylabel(
    r"$\sigma_{22}$ [MPa]",
    fontsize=16,
)

axes[1].set_title(
    "Normal Stress",
    fontsize=17,
    fontweight="bold",
)

axes[1].set_xlim(
    0.0,
    GAMMA_MAX,
)

axes[1].set_ylim(
    -0.02,
    0.2,
)

axes[1].grid(
    True,
    linestyle="--",
    alpha=0.5,
)

axes[1].tick_params(
    axis="both",
    labelsize=12,
)

plt.tight_layout()

configuration_label = str(
    CONFIGURATION
).replace("iso", "isotropic")

plt.savefig(
    f"stress_response_{configuration_label}_example.pdf",
    format="pdf",
    bbox_inches="tight",
)

plt.show()

### Goodness-of-fit metrics

Model performance is quantified using the coefficient of determination
$R^2$, root mean squared error (RMSE), mean absolute error (MAE), and the
scatter-normalized error $E_{\mathrm{scatter}}$.

The scatter-normalized error compares the model residuals with the measured
specimen-to-specimen experimental variability:

$
E_{\mathrm{scatter}}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}
\left(
\frac{y_i^{\mathrm{pred}}-\bar{y}_i^{\mathrm{exp}}}
{s_i^{\mathrm{exp}}}
\right)^2
}.
$

Points with zero or undefined experimental standard deviation are excluded
from the computation of $E_{\mathrm{scatter}}$.

In [ ]:
def scatter_normalized_error(
    y_exp,
    y_pred,
    y_std,
):
    """
    Compute the scatter-normalized root-mean-square residual.

    The metric is defined as

        E_scatter =
            sqrt(
                mean(
                    ((y_pred - y_exp) / y_std)**2
                )
            )

    where ``y_std`` is the specimen-to-specimen experimental standard
    deviation evaluated on the same shear-strain grid as ``y_exp``.

    Parameters
    ----------
    y_exp : array-like
        Experimental mean response.

    y_pred : array-like
        Model prediction.

    y_std : array-like
        Experimental standard deviation.

    Returns
    -------
    float
        Scatter-normalized error.

    Notes
    -----
    Data points with non-finite values or zero experimental standard
    deviation are excluded because the standardized residual is undefined
    at those locations.

    No artificial numerical regularization is added to the denominator.
    """
    y_exp = np.asarray(
        y_exp,
        dtype=float,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    ).reshape(-1)

    y_std = np.asarray(
        y_std,
        dtype=float,
    ).reshape(-1)

    mask = (
        np.isfinite(y_exp)
        & np.isfinite(y_pred)
        & np.isfinite(y_std)
        & (y_std > 0.0)
    )

    if not np.any(mask):
        return np.nan

    standardized_residuals = (
        y_pred[mask]
        - y_exp[mask]
    ) / y_std[mask]

    return float(
        np.sqrt(
            np.mean(
                standardized_residuals**2
            )
        )
    )

In [ ]:
def compute_fit_metrics(
    y_exp,
    y_pred,
    y_std,
):
    """
    Compute goodness-of-fit metrics for one stress component.

    Parameters
    ----------
    y_exp : array-like
        Experimental mean response.

    y_pred : array-like
        PA-CANN prediction.

    y_std : array-like
        Experimental specimen-to-specimen standard deviation.

    Returns
    -------
    dict
        Dictionary containing:

        ``r2``
            Coefficient of determination.

        ``rmse``
            Root mean squared error.

        ``mae``
            Mean absolute error.

        ``e_scatter``
            Scatter-normalized error.

    Notes
    -----
    If the experimental response has zero variance, ``R^2`` is undefined
    and ``numpy.nan`` is returned.
    """
    y_exp = np.asarray(
        y_exp,
        dtype=float,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    ).reshape(-1)

    if y_exp.shape != y_pred.shape:
        raise ValueError(
            "'y_exp' and 'y_pred' must have the same shape."
        )

    residuals = (
        y_pred
        - y_exp
    )

    rmse = np.sqrt(
        np.mean(
            residuals**2
        )
    )

    mae = np.mean(
        np.abs(
            residuals
        )
    )

    ss_res = np.sum(
        residuals**2
    )

    ss_tot = np.sum(
        (
            y_exp
            - np.mean(y_exp)
        )**2
    )

    if ss_tot > 0.0:
        r2 = (
            1.0
            - ss_res / ss_tot
        )
    else:
        r2 = np.nan

    e_scatter = scatter_normalized_error(
        y_exp=y_exp,
        y_pred=y_pred,
        y_std=y_std,
    )

    return {
        "r2": float(r2),
        "rmse": float(rmse),
        "mae": float(mae),
        "e_scatter": float(e_scatter),
    }


########################################################################################


def print_fit_metrics(
    configuration,
    tau_metrics,
    sigma_metrics,
):
    """
    Print PA-CANN goodness-of-fit metrics in a readable format.

    Parameters
    ----------
    configuration : str or int
        Material configuration or fiber orientation.

    tau_metrics : dict
        Metrics associated with the shear-stress response.

    sigma_metrics : dict
        Metrics associated with the normal-stress response.
    """
    if configuration == "iso":
        configuration_label = "Isotropic"
    else:
        configuration_label = (
            f"{configuration}°"
        )

    print("=" * 55)
    print(
        f"PA-CANN performance: "
        f"{configuration_label}"
    )
    print("=" * 55)

    print("Shear stress (tau_12)")
    print(
        f"  R^2       : "
        f"{tau_metrics['r2']:.4f}"
    )
    print(
        f"  RMSE      : "
        f"{tau_metrics['rmse']:.6f} MPa"
    )
    print(
        f"  MAE       : "
        f"{tau_metrics['mae']:.6f} MPa"
    )
    print(
        f"  E_scatter : "
        f"{tau_metrics['e_scatter']:.4f}"
    )

    print("")

    print("Normal stress (sigma_22)")
    print(
        f"  R^2       : "
        f"{sigma_metrics['r2']:.4f}"
    )
    print(
        f"  RMSE      : "
        f"{sigma_metrics['rmse']:.6f} MPa"
    )
    print(
        f"  MAE       : "
        f"{sigma_metrics['mae']:.6f} MPa"
    )
    print(
        f"  E_scatter : "
        f"{sigma_metrics['e_scatter']:.4f}"
    )

    print("=" * 55)

In [ ]:
tau_metrics = compute_fit_metrics(
    y_exp=tau_mean,
    y_pred=tau_pred,
    y_std=tau_std,
)

sigma_metrics = compute_fit_metrics(
    y_exp=sigma_mean,
    y_pred=sigma_pred,
    y_std=sigma_std,
)

print_fit_metrics(
    configuration=CONFIGURATION,
    tau_metrics=tau_metrics,
    sigma_metrics=sigma_metrics,
)

- ### Saving Results:

In [ ]:
# =============================================================================
# Export numerical results
# =============================================================================

output_file = (
    f"results_{configuration_label}_example.npz"
)

np.savez_compressed(
    output_file,

    # Configuration
    configuration=np.asarray(
        [str(CONFIGURATION)]
    ),

    # Experimental response
    gamma=gamma_plot,
    tau_mean=tau_mean,
    tau_std=tau_std,
    sigma_mean=sigma_mean,
    sigma_std=sigma_std,

    # Individual experimental specimen curves
    tau_curves=results["tau_curves"],
    sigma_curves=results["sigma_curves"],

    # PA-CANN predictions
    tau_pred=tau_pred,
    sigma_pred=sigma_pred,

    # Optimization history
    loss_history=loss_history,

    # Poynting-aware parameter histories
    zeta_vl_history=np.asarray(
        poynting_tracker.zeta_vl_history
    ),
    c2_history=np.asarray(
        poynting_tracker.c2_history
    ),
    c2_eff_0_history=np.asarray(
        poynting_tracker.c2_eff_0_history
    ),
    mu0_iso_history=np.asarray(
        poynting_tracker.mu0_iso_history
    ),
)

print(
    f"Results saved to: {output_file}"
)